# Visualise Embedding of ResNet with SimCLR Pretraining

In this notebook, we pass various data (train, test, generated) through the
pretrained ResNet backbone to get the 512-dimensional embeddings. Then, we
perform PCA on the embeddings to reduce the dimensionality to 2. We plot the
embeddings in 2D space. Interactive plots with Altair are also included.

The idea is that good generations should overlap with the train (and test) data,
while bad generations should form their own cluster further away. This is an
indicator that the learned embeddings are meaningful and can be used for
computing FRDS.

## Preliminaries

The repository assumes all entry points to be run as modules from root. Since
this notebook is 2 levels deep, the below block moves us to the root directory.

In [ ]:
import os

# Change this variable if the root folder name has been changed
root_dir = "nvae-shape-encoding"
current_dir = os.getcwd()

if not current_dir.endswith(root_dir):
    %cd ../..

assert os.getcwd().endswith(root_dir)

Update `model_path` to choose the model.

In [ ]:
import lightning as L
import torch

from arch.simclr.utils import load_simclr_backbone
from const import FRDS_MODEL_PATH, SEED
from data_modules.acdc import ACDCMaskDataModule
from utils.utils import setup_device

# Setup device
device = setup_device()
print(f"Device: {device}")

# Seed
L.seed_everything(SEED)

# Load data
data_module = ACDCMaskDataModule(batch_size=20)

# Reseed after preprocessing data
L.seed_everything(SEED)

# Load model
model = load_simclr_backbone(FRDS_MODEL_PATH)

Get the entire train and test data.

In [ ]:
data_train = data_module.data_train.masks
data_test = data_module.data_test.masks

print(data_train.shape)
print(data_test.shape)

## Encode Embeddings

We are now ready to pass the data through the ResNet backbone to get the 512-dimensional embeddings.

In [ ]:
from utils.eval import encode_embeddings

model = model.to(device)

# Extract features for train images
train_feats = encode_embeddings(data_train, model, device)
print(train_feats.shape)

# Extract features for test images
test_feats = encode_embeddings(data_test, model, device)
print(test_feats.shape)

We use PCA to reduce the 512-dimensional embeddings to 2.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
pca.fit(train_feats)

print(f"Explained variance: {pca.explained_variance_ratio_}")
print(f"Sum: {pca.explained_variance_ratio_.sum()}")

train_feats_reduced = pca.transform(train_feats)
test_feats_reduced = pca.transform(test_feats)

print(train_feats_reduced.shape)
print(test_feats_reduced.shape)

Extra: Out of curiosity, how's the explained variance if we use 10 components?

In [ ]:
pca_extend = PCA(n_components=10)
pca_extend.fit(train_feats)

print(f"Explained variance: {pca_extend.explained_variance_ratio_}")
print(f"Sum: {pca_extend.explained_variance_ratio_.sum()}")

## Plot Embeddings

First, let's do a simple scatter plot of the train and test data. Ideally, the
points from both datasets should overlap with no clear distinction. This would
be good as it means the model hasn't overfit to the train data.

In [ ]:
from matplotlib import pyplot as plt

plt.style.use("ggplot")

num_samples = min(train_feats_reduced.shape[0], test_feats_reduced.shape[0])

plt.figure(figsize=(5, 5))
plt.scatter(test_feats_reduced[:num_samples, 0], test_feats_reduced[:num_samples, 1], alpha=0.5, s=8, label="Test")
plt.scatter(train_feats_reduced[:num_samples, 0], train_feats_reduced[:num_samples, 1], alpha=0.5, s=8, label="Train")

plt.tight_layout()
plt.legend()
plt.show()

Let's do the same encode embeddings procedure for some generated data. We choose
2 models: (1) a model with good generations, and (2) a model with bad generations.

Update `vae_model_path` to choose the 2 models.

In [ ]:
from arch.nvae.nvae import NVAE
from arch.vae.vae import VAE
from utils.utils import discretise

num_samples = test_feats_reduced.shape[0]

def generate_data_vae(model: VAE) -> torch.Tensor:
    # Sample from latent space
    z = torch.randn(num_samples, model.hparams.latent_dim).to(device)

    with torch.no_grad():
        model.eval()
        model.to(device)
        
        # Generate segmentation maps from latent variables
        x_fake_logits: torch.Tensor = model.decoder(z)

    return x_fake_logits

def generate_data_nvae(model: NVAE) -> torch.Tensor:
    # Sample from latent space
    with torch.no_grad():
        model.eval()
        model.to(device)
        
        x_fake = model.decoder.generate(num_samples, device)
        feats_fake = model.conditional_coder(x_fake)

    return feats_fake

autoencoder_model_path = "logs/vae_acdc/info-vae/ld-8-beta-0-gamma-200/checkpoints/epoch=48-step=5243.ckpt"
autoencoder_model = VAE.load_from_checkpoint(autoencoder_model_path)

good_logits = generate_data_vae(autoencoder_model)

autoencoder_model_path = "logs/vae_acdc/info-vae/ld-8-beta-0-gamma-5/checkpoints/epoch=47-step=5136.ckpt"
autoencoder_model = VAE.load_from_checkpoint(autoencoder_model_path)

bad_logits = generate_data_vae(autoencoder_model)

good_fake_data = discretise(good_logits)
bad_fake_data = discretise(bad_logits)

print(good_fake_data.shape)
print(bad_fake_data.shape)

We view some example generated data from the 2 models. First image: test data. Second image: good generations. Third image: bad generations.

In [ ]:
from utils.utils import show_samples

# View generations
samples_idx = torch.randperm(data_test.shape[0])[:16]
test_samples = data_test[samples_idx]
test_samples = torch.argmax(test_samples, dim=1).unsqueeze(1)
show_samples(test_samples[:16], rgb=False, ncol=4, figsize=(4, 4))

good_generations = torch.argmax(good_logits, dim=1).unsqueeze(1)
show_samples(good_generations[:16], rgb=False, ncol=4, figsize=(4, 4))

bad_generations = torch.argmax(bad_logits, dim=1).unsqueeze(1)
show_samples(bad_generations[:16], rgb=False, ncol=4, figsize=(4, 4))

In [ ]:
good_feats = encode_embeddings(good_fake_data, model, device)
bad_feats = encode_embeddings(bad_fake_data, model, device)

good_feats_reduced = pca.transform(good_feats)
bad_feats_reduced = pca.transform(bad_feats)

print(good_feats_reduced.shape)
print(bad_feats_reduced.shape)

Now, let's also do a simple scatter plot of the generated data. Ideally, the
points from the good model should overlap with the train data, while the points
from the bad model should form a separate cluster. This would be good as it
means the SimCLR-pretrained model has learned to separate good quality masks
from bad quality masks.

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(test_feats_reduced[:, 0], test_feats_reduced[:, 1], alpha=0.5, s=8, label="Test")
plt.scatter(good_feats_reduced[:, 0], good_feats_reduced[:, 1], alpha=0.5, s=8, label="Good")

plt.tight_layout()
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(test_feats_reduced[:, 0], test_feats_reduced[:, 1], alpha=0.5, s=8, label="Test")
plt.scatter(bad_feats_reduced[:, 0], bad_feats_reduced[:, 1], alpha=0.5, s=8, label="Bad")

plt.tight_layout()
plt.legend()
plt.show()

Now, let's work towards an interactive plot with Altair to see the corresponding
segmentation masks and where they are embedded in the 2D space. We first need to
do some data processing.

In [ ]:
train_imgs = torch.argmax(data_train, dim=1).unsqueeze(1)
train_imgs = train_imgs.permute(0, 2, 3, 1).float()

test_imgs = torch.argmax(data_test, dim=1).unsqueeze(1)
test_imgs = test_imgs.permute(0, 2, 3, 1).float()

good_imgs = good_generations.permute(0, 2, 3, 1).float()
bad_imgs = bad_generations.permute(0, 2, 3, 1).float()

print(train_imgs.shape)
print(test_imgs.shape)
print(good_imgs.shape)
print(bad_imgs.shape)

We'll sample 40 points from each dataset only to (1) make the plot readable and (2) an
interactive plot with images takes a long time to render and may crash with more
load.

In [ ]:
import pandas as pd

dim_labels = ["dim-1", "dim-2"]

train_df = pd.DataFrame(train_feats_reduced, columns=dim_labels)
train_df["dataset"] = "train"
train_df["image"] = [tensor.numpy() for tensor in train_imgs]

test_df = pd.DataFrame(test_feats_reduced, columns=dim_labels)
test_df["dataset"] = "test"
test_df["image"] = [tensor.numpy() for tensor in test_imgs]

good_df = pd.DataFrame(good_feats_reduced, columns=dim_labels)
good_df["dataset"] = "good-gen-model"
good_df["image"] = [tensor.cpu().numpy() for tensor in good_imgs]

bad_df = pd.DataFrame(bad_feats_reduced, columns=dim_labels)
bad_df["dataset"] = "bad-gen-model"
bad_df["image"] = [tensor.cpu().numpy() for tensor in bad_imgs]

total_samples = 160

train_sampled_df = train_df.sample(n=total_samples // 4, replace=False)
test_sampled_df = test_df.sample(n=total_samples // 4, replace=False)
good_sampled_df = good_df.sample(n=total_samples // 4, replace=False)
bad_sampled_df = bad_df.sample(n=total_samples // 4, replace=False)

df = pd.concat([train_sampled_df, test_sampled_df, good_sampled_df, bad_sampled_df])

print(df.shape)
print(df["image"].iloc[0].shape)

df.head()

We show the interactive plot below.

**Do not run this cell multiple times as the notebook cache accumulates. The
plot alone takes a few hundred MB in memory, so it will crash if re-run multiple
times. Clear the outputs before restarting.**

In [ ]:

import altair as alt

selection = alt.selection_single(on="mouseover", nearest=True)

scatter_plot = alt.Chart(df).mark_circle(
    size=80,
    opacity=0.6,
).encode(
    x=alt.X(dim_labels[0]),
    y=alt.Y(dim_labels[1]),
    color=alt.condition(
        selection,
        alt.value("#000"),
        alt.Color("dataset"),
    ),
    tooltip=[dim_labels[0], dim_labels[1], "dataset"],
).add_selection(
    selection,
).properties(
    width=360,
    height=360,
)

# transform_filter ensures only the selected data points are processed to be visualised
masks_plot = alt.Chart(df).transform_filter(
    selection,
# Build the image from each pixel value
).transform_window(
    index="count()",
).transform_flatten(
    ["image"],
).transform_window(
    row="count()",
    groupby=["index"],
).transform_flatten(
    ["image"],
).transform_window(
    column="count()",
    groupby=["index", "row"],
).mark_rect(stroke="black",strokeWidth=0).encode(
    alt.X("column:O", axis=None),
    alt.Y("row:O", axis=None),
    alt.Color(
        "image:Q",
        scale=alt.Scale(
            scheme=alt.SchemeParams(
                "viridis",
                extent=[0, 1],
            ),
        ),
        legend=None,
    ),
).properties(
    width=360,
    height=360,
)

(scatter_plot | masks_plot).show()